delete all periods where <= 7 days are missing

In [3]:
import pandas as pd


In [114]:
phq8 = pd.read_csv("./data_cleaned/phq8_data.csv")

In [115]:
phq8.head()


,participant_name,timeInterval_ID,phq8_score
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,9
1,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1458,1
2,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1472,1
3,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1486,3
4,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1514,11


In [116]:
phq8.shape[0]

13122

In [117]:
sleep = pd.read_csv("./data_cleaned/sleep_data.csv")

In [4]:
total_dataset = pd.read_stata("./data/totaldataset.dta")

/var/folders/bt/f2dk62zd4lv80vqdphjgmw840000gn/T/ipykernel_22133/4226013150.py:1: UserWarning: Encountered %tC format. Leaving in Stata Internal Format.
  total_dataset = pd.read_stata("./data/totaldataset.dta")


In [119]:
sleep_obs = sleep[['participant_name', 'timeInterval_ID']].copy()
sleep_obs['observed'] = 1

In [120]:
sleep_obs = sleep_obs.rename(columns = {'timeInterval_ID': 'timeInterval_ID_sleep'})

In [121]:
window_size = 14

In [122]:
# Expand each PHQ-8 row into 14 prior days
window = phq8[['participant_name', 'timeInterval_ID']].copy()

window = window.loc[window.index.repeat(window_size)].copy()
window['lag'] = list(range(1, window_size+1)) * len(phq8)

# Compute the prior day IDs
window['timeInterval_ID_window'] = window['timeInterval_ID'] - window['lag']

In [123]:
window = window.merge(
    sleep_obs,
    left_on=['participant_name', 'timeInterval_ID_window'],
    right_on=['participant_name', 'timeInterval_ID_sleep'],
    how='left'
)

In [124]:
window['missing'] = window['observed'].isna().astype(int)

missing_counts = (
    window.groupby(['participant_name', 'timeInterval_ID'])
    ['missing']
    .sum()
    .reset_index(name=f'sleep_missing_last{window_size}')
)

In [125]:
phq8 = phq8.merge(
    missing_counts,
    on=['participant_name', 'timeInterval_ID'],
    how='left'
)

In [126]:
phq8

,participant_name,timeInterval_ID,phq8_score,sleep_missing_last14
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,9,14
1,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1458,1,4
2,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1472,1,3
3,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1486,3,5
4,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1514,11,4
...,...,...,...,...
13117,5f3217f1-c751-4fee-a8c4-21acc5153d58,1831,20,0
13118,5f3217f1-c751-4fee-a8c4-21acc5153d58,1845,20,0
13119,5f3217f1-c751-4fee-a8c4-21acc5153d58,1859,17,1
13120,6b52c798-bf6e-4512-af0d-98e39939fd37,1642,8,3


In [127]:
missing_threshold = 7 # up to 7 missing days in a period

In [128]:
phq8 = phq8[phq8[f"sleep_missing_last{window_size}"] <= missing_threshold]

In [79]:
phq8.shape[0]

8359

In [129]:
phq8

,participant_name,timeInterval_ID,phq8_score,sleep_missing_last14
1,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1458,1,4
2,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1472,1,3
3,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1486,3,5
4,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1514,11,4
5,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1528,5,5
...,...,...,...,...
13117,5f3217f1-c751-4fee-a8c4-21acc5153d58,1831,20,0
13118,5f3217f1-c751-4fee-a8c4-21acc5153d58,1845,20,0
13119,5f3217f1-c751-4fee-a8c4-21acc5153d58,1859,17,1
13120,6b52c798-bf6e-4512-af0d-98e39939fd37,1642,8,3


In [80]:
window = phq8[['participant_name', 'timeInterval_ID']].copy()

window = window.loc[window.index.repeat(window_size)].copy()
window['lag'] = list(range(1, window_size + 1)) * len(phq8)

window['timeInterval_ID_window'] = window['timeInterval_ID'] - window['lag']

In [81]:
window = window.merge(
    sleep[['participant_name', 'timeInterval_ID', 'total_sleep_time']],
    left_on=['participant_name', 'timeInterval_ID_window'],
    right_on=['participant_name', 'timeInterval_ID'],
    how='left'
)

In [82]:
window = window.rename(columns={'timeInterval_ID_x': 'timeInterval_ID'})

In [83]:
agg = (
    window.groupby(['participant_name', 'timeInterval_ID'])['total_sleep_time']
    .agg(
        sleep_min='min',
        sleep_max='max',
        sleep_mean='mean',
        sleep_median='median',
        sleep_std='std',
        sleep_q25=lambda x: x.quantile(0.25),
        sleep_q75=lambda x: x.quantile(0.75)
    )
    .reset_index()
)

# Compute IQR
agg['sleep_iqr'] = agg['sleep_q75'] - agg['sleep_q25']

# Drop intermediate quartiles if you want
agg = agg.drop(columns=['sleep_q25', 'sleep_q75'])

In [84]:
phq8_with_sleep_stats = phq8.merge(
    agg,
    on=['participant_name', 'timeInterval_ID'],
    how='left'
)

In [85]:
print(phq8_with_sleep_stats.head(10))

                       participant_name  timeInterval_ID  phq8_score  \
0  9078ae18-c644-4cdb-a582-6a2aa671ecb4             1458           1   
1  9078ae18-c644-4cdb-a582-6a2aa671ecb4             1472           1   
2  9078ae18-c644-4cdb-a582-6a2aa671ecb4             1486           3   
3  9078ae18-c644-4cdb-a582-6a2aa671ecb4             1514          11   
4  9078ae18-c644-4cdb-a582-6a2aa671ecb4             1528           5   
5  9078ae18-c644-4cdb-a582-6a2aa671ecb4             1542           7   
6  9078ae18-c644-4cdb-a582-6a2aa671ecb4             1556           7   
7  9078ae18-c644-4cdb-a582-6a2aa671ecb4             1598           7   
8  9078ae18-c644-4cdb-a582-6a2aa671ecb4             1416           0   
9  9078ae18-c644-4cdb-a582-6a2aa671ecb4             1612           0   

   sleep_min  sleep_max    sleep_mean  sleep_median    sleep_std  sleep_iqr  
0    10560.0    30390.0  24657.000000       26070.0  6000.957424     5962.5  
1    18240.0    26880.0  23045.454545       23490.0

In [131]:
for column in total_dataset.columns:
    print('\"' + column + '\",' )

"record_id",
"redcap_event_name",
"redcap_repeat_instrument",
"redcap_repeat_instance",
"subject_id",
"human_readable_id",
"radar_integration_complete",
"dob",
"age",
"gender",
"gender_at_birth",
"marital_status",
"living_together",
"children",
"no_of_children",
"children_live_with_you",
"youngest_child_age",
"ethnicity_1",
"white_other",
"mixed_other",
"asian_other",
"black_other",
"ethnic_other",
"country",
"eduction_finish_age",
"drop_school_1",
"qualifications",
"degree_1",
"benefits_type___0",
"benefits_type___1",
"benefits_type___2",
"benefits_type___3",
"benefits_type___4",
"benefits_type___5",
"benefits_type___6",
"benefits_type___7",
"benefits_type___8",
"benefits_type___9",
"benefits_type___10",
"benefits_type_other",
"benefits",
"income_gbp",
"income_contribution",
"accommodation",
"accommodation_satisfaction",
"utilities___0",
"utilities___1",
"utilities___2",
"utilities___3",
"utilities___4",
"utilities___5",
"utilities___6",
"utilities___7",
"utilities___8",
"utilities___

totaldataset exploration

In [6]:
duplicates = total_dataset.duplicated(subset=["subject_id"])


In [7]:
duplicates

0       False
1        True
2        True
3        True
4        True
        ...  
3772     True
3773    False
3774     True
3775     True
3776     True
Length: 3777, dtype: bool

In [132]:
total_dataset["subject_id"].head()

0    007751c5-d7ad-4bec-a58f-abf32500e2ae
1    007751c5-d7ad-4bec-a58f-abf32500e2ae
2    007751c5-d7ad-4bec-a58f-abf32500e2ae
3    007751c5-d7ad-4bec-a58f-abf32500e2ae
4    007751c5-d7ad-4bec-a58f-abf32500e2ae
Name: subject_id, dtype: object

In [144]:
total_dataset["asian_other"]

0        
1        
2        
3        
4        
       ..
3772     
3773     
3774     
3775     
3776     
Name: asian_other, Length: 3777, dtype: object

In [143]:
total_dataset[~total_dataset["ethnicity_1"].isna()]["ethnicity_1"].head(20)

10                                             Dutch
14     English/Welsh/Scottish/Northern Irish/British
23                                             Dutch
34     English/Welsh/Scottish/Northern Irish/British
40                                             Dutch
45                                       Bangladeshi
52                                             Dutch
65     English/Welsh/Scottish/Northern Irish/British
73     English/Welsh/Scottish/Northern Irish/British
92     English/Welsh/Scottish/Northern Irish/British
93                                             Dutch
100    English/Welsh/Scottish/Northern Irish/British
105    English/Welsh/Scottish/Northern Irish/British
114    English/Welsh/Scottish/Northern Irish/British
118    English/Welsh/Scottish/Northern Irish/British
127    English/Welsh/Scottish/Northern Irish/British
136    English/Welsh/Scottish/Northern Irish/British
138                                            Dutch
145                       Any other White back

In [148]:
total_dataset["accommodation_satisfaction"]

0       Fairly satisfied
1                    NaN
2                    NaN
3                    NaN
4                    NaN
              ...       
3772                 NaN
3773    Fairly satisfied
3774                 NaN
3775                 NaN
3776                 NaN
Name: accommodation_satisfaction, Length: 3777, dtype: category
Categories (5, object): ['Very dissatisfied' < 'Slightly dissatisfied' < 'Neither satisfied nor dissatisfied' < 'Fairly satisfied' < 'Very satisfied']

In [152]:
total_dataset_clean = pd.read_csv("./data_cleaned/totaldataset.csv")

Steps for pipeline:

1. demographics data needs to be preprocessed
    a. imputation
    b. scaling?

2. physiological measurement data needs to be preprocessed
    a. scaling

In nested cross validation, imputers and scalars must only be fit on the training data and then used to transform the validation and test data
Nested cross validation split on participant level